<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/01_data_inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01_data_inventory

Цель ноутбука:
- подключить Google Drive;
- проверить структуру папок проекта;
- провести инвентаризацию сырых данных;
- сохранить сводный отчёт по датасетам;
- зафиксировать, какие источники уже готовы к дальнейшей обработке.

Ожидаемые папки сырых данных:
- `rusentiment`
- `eedi_tutoring`
- `esconv`
- `student_feedback`
- `project_team_analog`

In [1]:
!pip -q install datasets pandas numpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
from pathlib import Path
from datetime import datetime

import pandas as pd

In [5]:
# пути

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

ADMIN_DIR = DRIVE_PROJECT_ROOT / "00_admin"
RAW_DIR = DRIVE_PROJECT_ROOT / "01_data_raw"
INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"
LABELING_DIR = DRIVE_PROJECT_ROOT / "03_labeling"
READY_DIR = DRIVE_PROJECT_ROOT / "04_datasets_ready"
MODELS_DIR = DRIVE_PROJECT_ROOT / "05_models"
REPORTS_DIR = DRIVE_PROJECT_ROOT / "06_reports"
EXPORTS_DIR = DRIVE_PROJECT_ROOT / "07_exports"
ARCHIVE_DIR = DRIVE_PROJECT_ROOT / "08_archive"

REPORTS_TABLES_DIR = REPORTS_DIR / "tables_for_coursework"
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_PROJECT_ROOT =", DRIVE_PROJECT_ROOT)
print("RAW_DIR            =", RAW_DIR)
print("REPORTS_TABLES_DIR =", REPORTS_TABLES_DIR)

DRIVE_PROJECT_ROOT = /content/drive/MyDrive/tutors_sentiment_project
RAW_DIR            = /content/drive/MyDrive/tutors_sentiment_project/01_data_raw
REPORTS_TABLES_DIR = /content/drive/MyDrive/tutors_sentiment_project/06_reports/tables_for_coursework


In [6]:
# ожидаемые датасеты

EXPECTED_DATASETS = {
    "rusentiment": {
        "role": "базовый русскоязычный корпус тональности",
        "expected_use": "базовое дообучение / baseline",
    },
    "eedi_tutoring": {
        "role": "предметно близкие реальные тьюторские диалоги",
        "expected_use": "ручная доменная разметка / тестирование",
    },
    "esconv": {
        "role": "диалоги эмоциональной поддержки",
        "expected_use": "аналог поддерживающей коммуникации",
    },
    "student_feedback": {
        "role": "образовательные отзывы студентов",
        "expected_use": "дополнительный образовательный корпус",
    },
    "project_team_analog": {
        "role": "аналог коммуникации проектной команды",
        "expected_use": "дополнительный внешний аналог",
    },
}

pd.DataFrame.from_dict(EXPECTED_DATASETS, orient="index")

,role,expected_use
rusentiment,базовый русскоязычный корпус тональности,базовое дообучение / baseline
eedi_tutoring,предметно близкие реальные тьюторские диалоги,ручная доменная разметка / тестирование
esconv,диалоги эмоциональной поддержки,аналог поддерживающей коммуникации
student_feedback,образовательные отзывы студентов,дополнительный образовательный корпус
project_team_analog,аналог коммуникации проектной команды,дополнительный внешний аналог


In [7]:
def get_file_stats(folder: Path):
    if not folder.exists():
        return {
            "exists": False,
            "num_files": 0,
            "num_dirs": 0,
            "total_size_mb": 0.0,
            "extensions": "",
            "sample_files": [],
        }

    files = [p for p in folder.rglob("*") if p.is_file()]
    dirs = [p for p in folder.rglob("*") if p.is_dir()]

    total_size = sum(p.stat().st_size for p in files) / (1024 * 1024)
    exts = sorted({p.suffix.lower() if p.suffix else "<no_ext>" for p in files})

    return {
        "exists": True,
        "num_files": len(files),
        "num_dirs": len(dirs),
        "total_size_mb": round(total_size, 2),
        "extensions": ", ".join(exts),
        "sample_files": [str(p.relative_to(folder)) for p in files[:10]],
    }


def list_top_level(folder: Path):
    if not folder.exists():
        return []
    return sorted([p.name for p in folder.iterdir()])


def print_tree(folder: Path, max_depth=2, prefix=""):
    if not folder.exists():
        print(f"{folder} — не существует")
        return

    def _walk(path, depth, indent=""):
        if depth > max_depth:
            return
        items = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
        for item in items:
            print(indent + item.name + ("/" if item.is_dir() else ""))
            if item.is_dir():
                _walk(item, depth + 1, indent + "    ")

    print(folder)
    _walk(folder, 0)

In [8]:
inventory_rows = []

for dataset_name, meta in EXPECTED_DATASETS.items():
    folder = RAW_DIR / dataset_name
    stats = get_file_stats(folder)

    inventory_rows.append({
        "dataset_name": dataset_name,
        "folder": str(folder),
        "exists": stats["exists"],
        "num_files": stats["num_files"],
        "num_dirs": stats["num_dirs"],
        "total_size_mb": stats["total_size_mb"],
        "extensions": stats["extensions"],
        "role": meta["role"],
        "expected_use": meta["expected_use"],
    })

inventory_df = pd.DataFrame(inventory_rows)
inventory_df

,dataset_name,folder,exists,num_files,num_dirs,total_size_mb,extensions,role,expected_use
0,rusentiment,/content/drive/MyDrive/tutors_sentiment_projec...,True,0,0,0.0,,базовый русскоязычный корпус тональности,базовое дообучение / baseline
1,eedi_tutoring,/content/drive/MyDrive/tutors_sentiment_projec...,True,0,0,0.0,,предметно близкие реальные тьюторские диалоги,ручная доменная разметка / тестирование
2,esconv,/content/drive/MyDrive/tutors_sentiment_projec...,True,0,0,0.0,,диалоги эмоциональной поддержки,аналог поддерживающей коммуникации
3,student_feedback,/content/drive/MyDrive/tutors_sentiment_projec...,True,0,0,0.0,,образовательные отзывы студентов,дополнительный образовательный корпус
4,project_team_analog,/content/drive/MyDrive/tutors_sentiment_projec...,True,0,0,0.0,,аналог коммуникации проектной команды,дополнительный внешний аналог


In [9]:
inventory_csv_path = REPORTS_TABLES_DIR / "data_inventory_summary.csv"
inventory_json_path = ADMIN_DIR / "data_inventory_summary.json"

inventory_df.to_csv(inventory_csv_path, index=False, encoding="utf-8-sig")

with open(inventory_json_path, "w", encoding="utf-8") as f:
    json.dump(inventory_rows, f, ensure_ascii=False, indent=2)

print("CSV сохранён:", inventory_csv_path)
print("JSON сохранён:", inventory_json_path)

CSV сохранён: /content/drive/MyDrive/tutors_sentiment_project/06_reports/tables_for_coursework/data_inventory_summary.csv
JSON сохранён: /content/drive/MyDrive/tutors_sentiment_project/00_admin/data_inventory_summary.json


In [10]:
for dataset_name in EXPECTED_DATASETS.keys():
    folder = RAW_DIR / dataset_name
    print("=" * 70)
    print(f"[{dataset_name}]")
    if folder.exists():
        print("Верхний уровень:", list_top_level(folder))
    else:
        print("Папка отсутствует")

[rusentiment]
Верхний уровень: []
[eedi_tutoring]
Верхний уровень: []
[esconv]
Верхний уровень: []
[student_feedback]
Верхний уровень: []
[project_team_analog]
Верхний уровень: []


In [11]:
dataset_to_inspect = "student_feedback"
print_tree(RAW_DIR / dataset_to_inspect, max_depth=3)

/content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback


In [12]:
# 1. ESConv
esconv_json = RAW_DIR / "esconv" / "ESConv.json"
if esconv_json.exists():
    with open(esconv_json, "r", encoding="utf-8") as f:
        esconv_data = json.load(f)

    print("ESConv:")
    print("  количество диалогов:", len(esconv_data))
    if len(esconv_data) > 0:
        print("  ключи первого объекта:", list(esconv_data[0].keys()))
else:
    print("ESConv.json не найден")

ESConv.json не найден


In [13]:
# 2. Student Feedback
student_feedback_tsv = list((RAW_DIR / "student_feedback").rglob("*.tsv"))
print("Student Feedback TSV files:", len(student_feedback_tsv))
for p in student_feedback_tsv[:10]:
    print(" ", p)

Student Feedback TSV files: 0


In [15]:
# 3. CSV / JSON / Parquet во всех сырых папках
all_interesting = []
for dataset_name in EXPECTED_DATASETS.keys():
    folder = RAW_DIR / dataset_name
    if folder.exists():
        for ext in ["*.csv", "*.json", "*.parquet", "*.tsv"]:
            for p in folder.rglob(ext):
                all_interesting.append({
                    "dataset_name": dataset_name,
                    "file_path": str(p),
                    "suffix": p.suffix.lower(),
                    "size_kb": round(p.stat().st_size / 1024, 2),
                })

interesting_df = pd.DataFrame(all_interesting)
interesting_df.head(20)

""


In [16]:
interesting_csv_path = REPORTS_TABLES_DIR / "raw_interesting_files.csv"
interesting_df.to_csv(interesting_csv_path, index=False, encoding="utf-8-sig")
print("Сохранено:", interesting_csv_path)

Сохранено: /content/drive/MyDrive/tutors_sentiment_project/06_reports/tables_for_coursework/raw_interesting_files.csv


In [17]:
def readiness_label(row):
    if not row["exists"]:
        return "нет папки"
    if row["num_files"] == 0:
        return "папка пустая"
    return "готов к следующему этапу"

inventory_df["status"] = inventory_df.apply(readiness_label, axis=1)
inventory_df[["dataset_name", "exists", "num_files", "extensions", "status"]]

,dataset_name,exists,num_files,extensions,status
0,rusentiment,True,0,,папка пустая
1,eedi_tutoring,True,0,,папка пустая
2,esconv,True,0,,папка пустая
3,student_feedback,True,0,,папка пустая
4,project_team_analog,True,0,,папка пустая


In [19]:
print("=" * 70)
print("Инвентаризация данных завершена")
print("=" * 70)
print("Всего ожидаемых источников:", len(EXPECTED_DATASETS))
print("Есть папки:", int(inventory_df["exists"].sum()))
print("Есть непустые папки:", int((inventory_df["num_files"] > 0).sum()))

Инвентаризация данных завершена
Всего ожидаемых источников: 5
Есть папки: 5
Есть непустые папки: 0
